In [2]:
import pandas as pd
import numpy as np

# Load the existing inspection data
df = pd.read_csv('moat_scores_inspection.csv')

# 1. Handle Negative Values for Power BI Visuals
# For margins, anything below -100% (-1.0) is effectively 'distressed'.
# Capping prevents outliers from squishing the chart axes.
margin_cols = ['FourYearNetProfitMargin', 'FourYearOperatingProfitMargin', 'FourYearReturnOnEquity', 'FourYearReturnOnAssets']
for col in margin_cols:
    df[col] = df[col].clip(lower=-1.0)

# 2. Add a 'Financial Health' flag for negative values
# This helps with filtering in Power BI
df['Equity_Status'] = np.where(df['FourYearTotalDebtToEquity'] < 0, 'Negative Equity', 'Positive Equity')

# 3. Handle Interest Coverage (Negative means the company isn't covering interest)
df['Interest_Coverage_Status'] = np.where(df['FourYearInterestCoverage'] < 0, 'Non-Covering', 'Healthy')

# Save the cleaned version
cleaned_filename = 'cleaned_moat_data.csv'
df.to_csv(cleaned_filename, index=False)

print(f"Cleaned data saved to {cleaned_filename}")
print(df[['Ticker', 'FourYearOperatingProfitMargin', 'Equity_Status']].head())

Cleaned data saved to cleaned_moat_data.csv
  Ticker  FourYearOperatingProfitMargin    Equity_Status
0   AAPL                       0.308978  Positive Equity
1   AMZN                       0.062104  Positive Equity
2   AVGO                       0.397083  Positive Equity
3   BMNR                      -1.000000  Positive Equity
4   COST                       0.035510  Positive Equity


In [ ]:
import pandas as pd
import numpy as np
from scipy import stats

# Load the cleaned data
df = pd.read_csv('cleaned_moat_data.csv')

# Compute mean of Four Year Operating Profit Margin
mean_operating_margin = df['FourYearOperatingProfitMargin'].mean()

# Compute mode of Four Year Operating Profit Margin
# Method 1: Using pandas mode() (returns all modes)
mode_operating_margin_pandas = df['FourYearOperatingProfitMargin'].mode()

# Method 2: Using scipy stats mode() (returns first mode)
mode_operating_margin_scipy = stats.mode(df['FourYearOperatingProfitMargin'], keepdims=True)

# Print results
print(f"Mean Four Year Operating Profit Margin: {mean_operating_margin:.4f}")
print(f"Mode(s) from pandas: {mode_operating_margin_pandas.values}")
print(f"Mode from scipy: {mode_operating_margin_scipy.mode[0]:.4f} (frequency: {mode_operating_margin_scipy.count[0]})")

# Optional: Get mode with frequency count using pandas
mode_freq = df['FourYearOperatingProfitMargin'].value_counts().head(1)
print(f"\nMost frequent value: {mode_freq.index[0]:.4f} (appears {mode_freq.values[0]} times)")

# If you want to store them for further use:
mean_value = mean_operating_margin
mode_value = mode_operating_margin_pandas.iloc[0] if len(mode_operating_margin_pandas) > 0 else np.nan